In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Cargar la matriz
ruta = r"C:\Users\hugod\costco-expansion-analysis\data\processed\matriz_espacial_fase2.csv"
df = pd.read_csv(ruta)

# 2. Ingeniería de Variables
# Estandarizamos para evitar el colapso matemático
df['ingreso_miles'] = df['ingreso_promedio_hogar'] / 1000
df['b2b_miles'] = df['masa_negocios_b2b'] / 1000
df['poblacion_cien_miles'] = df['poblacion_hogares'] / 100000

# NUEVA VARIABLE Y: ¿Existe inversión comprobada de un Club de Precios aquí?
# Si hay 1 o más competidores, el mercado ya demostró ser viable empíricamente.
df['Y_mercado_probado'] = np.where(df['total_competidores'] > 0, 1, 0)

# Definimos X (Sin incluir a los competidores, porque eso es lo que estamos prediciendo indirectamente)
variables_explicativas = ['ingreso_miles', 'b2b_miles', 'poblacion_cien_miles']
X = df[variables_explicativas]
X = sm.add_constant(X)
Y = df['Y_mercado_probado']

# 3. Entrenar el Modelo Logit
print("Entrenando Modelo Econométrico Logit (Comportamiento del Mercado)...\n")
try:
    modelo_logit = sm.Logit(Y, X).fit(disp=False)
    
    # 4. Resultados
    print("\n" + "="*80)
    print("REPORTE ECONOMÉTRICO: DETERMINANTES DE INVERSIÓN RETAIL (Fase 2)")
    print("="*80)
    print(modelo_logit.summary())
    
    # 5. Predicción de Probabilidades (El Radar de Inversión)
    # Aquí calculamos qué probabilidad le da el modelo a cada municipio
    df['probabilidad_teorica'] = modelo_logit.predict(X)
    
    # Filtramos los Océanos Azules: Alta probabilidad de éxito, pero CERO competencia actual
    oportunidades = df[(df['probabilidad_teorica'] > 0.40) & (df['total_competidores'] == 0)]
    
    print("\n" + "="*80)
    print("🎯 TOP MUNICIPIOS 'OCÉANO AZUL' PARA NUEVO COSTCO")
    print("(Alta demanda y capacidad B2B, pero cero saturación actual)")
    print("="*80)
    display(oportunidades[['ubica_geo', 'ingreso_promedio_hogar', 'masa_negocios_b2b', 'probabilidad_teorica']].sort_values(by='probabilidad_teorica', ascending=False))

except Exception as e:
    print(f"Error en la estimación: {e}")

Entrenando Modelo Econométrico Logit (Comportamiento del Mercado)...


REPORTE ECONOMÉTRICO: DETERMINANTES DE INVERSIÓN RETAIL (Fase 2)
                           Logit Regression Results                           
Dep. Variable:      Y_mercado_probado   No. Observations:                   98
Model:                          Logit   Df Residuals:                       94
Method:                           MLE   Df Model:                            3
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                  0.5144
Time:                        18:22:36   Log-Likelihood:                -23.405
converged:                       True   LL-Null:                       -48.196
Covariance Type:            nonrobust   LLR p-value:                 9.806e-11
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -4.7645      1.166     -4.086 

,ubica_geo,ingreso_promedio_hogar,masa_negocios_b2b,probabilidad_teorica
36,14101,68376.161543,4773,0.861556
58,19021,73976.541204,4116,0.805404
65,19031,70300.266333,4043,0.662112


In [3]:
"""Auditoría completa del proyecto Costco."""

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from nbformat import read

print('=' * 90)
print('AUDITORÍA COMPLETA: PROYECTO COSTCO MÉXICO')
print('=' * 90)

notebooks_dir = Path(r'C:\Users\hugod\costco-expansion-analysis\notebooks')
notebooks = sorted(
    path for path in notebooks_dir.glob('*.ipynb')
    if not path.name.startswith('.')
)
print(f'\nNOTEBOOKS ENCONTRADOS: {len(notebooks)}')
for number, notebook_path in enumerate(notebooks, 1):
    print(f'{number}. {notebook_path.name}')

auditorias = {}
for notebook_path in notebooks:
    print(f'\nANALIZANDO: {notebook_path.name}')
    try:
        with open(notebook_path, encoding='utf-8') as file:
            notebook = read(file, as_version=4)
        code = '\n'.join(
            cell.source for cell in notebook.cells if cell.cell_type == 'code'
        )
        libraries = {
            name for name, patterns in {
                'pandas': ('import pandas', 'import pd'),
                'numpy': ('import numpy', 'import np'),
                'statsmodels': ('import statsmodels', 'import sm'),
                'matplotlib/seaborn': ('import matplotlib', 'import seaborn'),
                'sklearn': ('import sklearn', 'from sklearn'),
            }.items() if any(pattern in code for pattern in patterns)
        }
        data_files = sorted({
            Path(match).name for match in re.findall(
                r"read_csv\(\s*r?['\"]([^'\"]+)", code
            )
        })
        output_files = sorted({
            Path(match).name for match in re.findall(
                r"(?:to_csv|savefig)\(\s*r?['\"]([^'\"]+)", code
            )
        })
        analyses = [
            description for keyword, description in {
                'OLS': 'Regresión OLS',
                'logit': 'Regresión logística',
                'describe()': 'Estadísticas descriptivas',
                'hist': 'Histogramas',
                'scatter': 'Gráficos scatter',
                'boxplot': 'Box plots',
                'merge': 'Unión de tablas',
                'groupby': 'Agregaciones por grupos',
            }.items() if keyword.lower() in code.lower()
        ]
        print(f'   Celdas: {len(notebook.cells)} ({sum(c.cell_type == "code" for c in notebook.cells)} de código)')
        print(f'   Librerías: {", ".join(sorted(libraries)) or "ninguna"}')
        print(f'   Datos: {", ".join(data_files) or "ninguno detectado"}')
        print(f'   Salidas: {", ".join(output_files) or "ninguna detectada"}')
        print(f'   Análisis: {", ".join(analyses) or "ninguno"}')
        auditorias[notebook_path.name] = {
            'celdas_totales': len(notebook.cells),
            'celdas_codigo': sum(c.cell_type == 'code' for c in notebook.cells),
            'librerias': sorted(libraries),
            'datos_entrada': data_files,
            'datos_salida': output_files,
            'analisis': analyses,
            'ruta': str(notebook_path),
        }
    except Exception as error:
        print(f'   ERROR: {error}')

processed_dir = Path(r'C:\Users\hugod\costco-expansion-analysis\data\processed')
print('\nARCHIVOS DE DATOS PROCESADOS:')
for csv_path in sorted(processed_dir.glob('*.csv')):
    data = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)
    print(f'   {csv_path.name}: {len(data):,} filas | {len(data.columns)} columnas')

results_dir = Path(r'C:\Users\hugod\costco-expansion-analysis\resultados')
print('\nRESULTADOS GENERADOS:')
for result_path in sorted(path for path in results_dir.rglob('*') if path.is_file()):
    print(f'   {result_path.name}: {result_path.stat().st_size / 1024:.1f} KB')

modelo_path = processed_dir / 'datos_modelo_final.csv'
if modelo_path.exists():
    df_modelo = pd.read_csv(modelo_path, encoding='utf-8-sig', low_memory=False)
    df_modelo.columns = df_modelo.columns.str.replace('ï»¿', '', regex=False)
    for column in ('ingreso_total', 'est_socio', 'tam_loc'):
        if column in df_modelo.columns:
            df_modelo[column] = pd.to_numeric(df_modelo[column], errors='coerce')
    print('\nESTADÍSTICAS CLAVE:')
    print(f'   Hogares válidos: {len(df_modelo):,}')
    print(f'   Ingreso promedio: ${df_modelo["ingreso_total"].mean():,.0f} MXN')
    print(f'   Ingreso mediano: ${df_modelo["ingreso_total"].median():,.0f} MXN')

print('\nPENDIENTE PARA SAR-LOGIT:')
print('   Integrar DENUE, variables macroeconómicas, matriz espacial W y una variable binaria municipal.')

output_path = results_dir / 'AUDITORIA_PROYECTO.json'
with open(output_path, 'w', encoding='utf-8') as file:
    json.dump(auditorias, file, indent=2, ensure_ascii=False)
print(f'\nAUDITORÍA GUARDADA EN: {output_path}')

AUDITORÍA COMPLETA: PROYECTO COSTCO MÉXICO

NOTEBOOKS ENCONTRADOS: 7
1. 01_exploracion_datos.ipynb
2. 01_exploracion_datos_BACKUP.ipynb
3. 02_modelo_econometrico_final.ipynb
4. 03_extraccion_denue.ipynb
5. 03b_inyectar_vehiculos.ipynb
6. 04_modelo_logit.ipynb
7. Untitled-1.ipynb

ANALIZANDO: 01_exploracion_datos.ipynb
   Celdas: 11 (10 de código)
   Librerías: matplotlib/seaborn, numpy, pandas, sklearn, statsmodels
   Datos: conjunto_de_datos_ingresos_enigh2022_ns.csv, datos_modelo.csv, enigh_monterrey_guadalajara_queretaro.csv, ingresos_por_hogar.csv
   Salidas: datos_modelo.csv, datos_modelo_final.csv, exploracion_modelo.png, ingresos_por_hogar.csv
   Análisis: Regresión OLS, Estadísticas descriptivas, Histogramas, Gráficos scatter, Box plots, Unión de tablas, Agregaciones por grupos

ANALIZANDO: 01_exploracion_datos_BACKUP.ipynb
   Celdas: 7 (6 de código)
   Librerías: numpy, pandas, sklearn, statsmodels
   Datos: conjunto_de_datos_ingresos_enigh2022_ns.csv, datos_modelo.csv, enigh_